# 02_Voice Feature Extraction

## Objective
The objective of this notebook is to extract informative voice features for recognition modeling.

## Method
Acoustic feature extraction workflow.

## Input
Preprocessed/standardized voice recordings.

## Processing Tasks

1. Load & Validate Voice Dataset.
2. Audio Feature Extraction Configuration.
3. MFCC Feature Extraction.
4. Zero Crossing Rate (ZCR) Feature Extraction.
5. RMS Energy Feature Extraction.
6. Combine Voice Features.

## Output
- Voice feature vectors.
- Labeled voice feature dataset.
- Exported features for model training.

## Purpose
Prepare high-quality voice features for model training and evaluation.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import librosa

import matplotlib.pyplot as plt

PROJECT_PATH = Path(r"E:\Multimodal Biometric Verification System")

INPUT_PATH = (
    PROJECT_PATH / "Dataset" / "Processed" / "Voice" / "Audio_Standardized"
)

OUTPUT_PATH = (
    PROJECT_PATH / "Dataset" / "Processed" / "Voice" / "Voice_Features"
)

USERS = ["User_01", "User_02"]

SAMPLE_RATE = 16000

N_MFCC = 13
N_FFT = 512
HOP_LENGTH = 160
WIN_LENGTH = 400

WINDOW = "hann"

print("Voice Feature Extraction Environment Configured.")
print(f"Input Path: {INPUT_PATH}")
print(f"Output Path: {OUTPUT_PATH}")
print(f"Sample Rate: {SAMPLE_RATE} Hz")
print(f"Number of MFCCs: {N_MFCC}")

# 1. Load & Validate Voice Dataset

This section loads all standardized voice recordings and validates their metadata.

The validation includes:
- Number of recordings per user.
- Audio file names.
- Sample rates.
- Number of channels.
- Recording durations.
- Missing and invalid values.


In [ ]:
audio_records = []

for user in USERS:

    user_path = INPUT_PATH / user

    audio_files = sorted(user_path.glob("*.wav"))

    print(f"{user}: {len(audio_files)} recordings found.")

    for audio_file in audio_files:

        audio_info = sf.info(audio_file)

        audio_records.append({
            "User_ID": user,
            "Audio_Name": audio_file.name,
            "Sample_Rate": audio_info.samplerate,
            "Channels": audio_info.channels,
            "Duration": audio_info.duration,
            "Frames": audio_info.frames
        })

voice_dataset = pd.DataFrame(audio_records)

print("\nTotal Audio Recordings:", len(voice_dataset))

print("\nMissing Values:")
print(voice_dataset.isnull().sum())

print("\nInvalid Audio Metadata:")
print(
    (
        (voice_dataset["Sample_Rate"] != SAMPLE_RATE)
        | (voice_dataset["Channels"] != 1)
        | (voice_dataset["Duration"] <= 0)
    ).sum()
)

display(voice_dataset.head(10))

# 2. Audio Feature Extraction Configuration

This section defines the parameters used to extract MFCC, Zero Crossing Rate (ZCR), and Root Mean Square Energy (RMS) features.

The selected parameters are configured for speech signal analysis at a sample rate of 16 kHz.


In [ ]:
# MFCC Configuration
MFCC_CONFIG = {
    "n_mfcc": N_MFCC,
    "n_fft": N_FFT,
    "hop_length": HOP_LENGTH,
    "win_length": WIN_LENGTH,
    "window": WINDOW
}

# ZCR Configuration (Zero-Crossing Rate)
ZCR_CONFIG = {
    "frame_length": WIN_LENGTH,
    "hop_length": HOP_LENGTH
}

# RMS Configuration
RMS_CONFIG = {
    "frame_length": WIN_LENGTH,
    "hop_length": HOP_LENGTH
}

print("Audio Feature Extraction Configuration")
print("-" * 45)

print("\nMFCC Configuration:")
for key, value in MFCC_CONFIG.items():
    print(f"{key}: {value}")

print("\nZCR Configuration:")
for key, value in ZCR_CONFIG.items():
    print(f"{key}: {value}")

print("\nRMS Configuration:")
for key, value in RMS_CONFIG.items():
    print(f"{key}: {value}")

print("\nWindow Duration:",
      f"{WIN_LENGTH / SAMPLE_RATE * 1000:.2f} ms")

print("Hop Duration:",
      f"{HOP_LENGTH / SAMPLE_RATE * 1000:.2f} ms")

print("FFT Duration:",
      f"{N_FFT / SAMPLE_RATE * 1000:.2f} ms")

# 3. MFCC Feature Extraction

This section extracts 13 Mel-Frequency Cepstral Coefficients (MFCCs) from each audio recording.

For each MFCC coefficient, the mean and standard deviation are calculated across all frames.

This produces a fixed-length feature vector of 26 MFCC features per recording.


In [ ]:
mfcc_records = []

for user in USERS:

    user_path = INPUT_PATH / user

    audio_files = sorted(user_path.glob("*.wav"))

    for audio_file in audio_files:

        signal, sample_rate = sf.read(audio_file)

        mfcc_features = librosa.feature.mfcc(
            y=signal,
            sr=sample_rate,
            **MFCC_CONFIG
        )

        mfcc_mean = np.mean(mfcc_features, axis=1)
        mfcc_std = np.std(mfcc_features, axis=1)

        record = {
            "User_ID": user,
            "Audio_Name": audio_file.name
        }

        for i in range(N_MFCC):

            record[f"MFCC_{i}_Mean"] = mfcc_mean[i]
            record[f"MFCC_{i}_Std"] = mfcc_std[i]

        mfcc_records.append(record)

mfcc_df = pd.DataFrame(mfcc_records)

print("MFCC Feature Extraction Completed")
print("-" * 45)

print("Total Recordings:", len(mfcc_df))
print("MFCC Features per Recording:", N_MFCC * 2)
print("Dataset Shape:", mfcc_df.shape)

display(mfcc_df.head())

# 4. Zero Crossing Rate (ZCR) Feature Extraction

This section extracts the Zero Crossing Rate (ZCR) from each audio recording.

For each recording, the mean and standard deviation of ZCR are calculated across all frames.

This produces two ZCR features per recording.


In [ ]:
zcr_records = []

for user in USERS:

    user_path = INPUT_PATH / user

    audio_files = sorted(user_path.glob("*.wav"))

    for audio_file in audio_files:

        signal, sample_rate = sf.read(audio_file)

        zcr_features = librosa.feature.zero_crossing_rate(
            y=signal,
            **ZCR_CONFIG
        )

        zcr_mean = np.mean(zcr_features)
        zcr_std = np.std(zcr_features)

        zcr_records.append({
            "User_ID": user,
            "Audio_Name": audio_file.name,
            "ZCR_Mean": zcr_mean,
            "ZCR_Std": zcr_std
        })

zcr_df = pd.DataFrame(zcr_records)

print("ZCR Feature Extraction Completed")
print("-" * 45)

print("Total Recordings:", len(zcr_df))
print("ZCR Features per Recording:", 2)
print("Dataset Shape:", zcr_df.shape)

display(zcr_df.head())

# 5. RMS Energy Feature Extraction

This section extracts Root Mean Square (RMS) energy features from each audio recording.

For each recording, the mean and standard deviation of RMS energy are calculated across all frames.

This produces two RMS features per recording.


In [ ]:
rms_records = []

for user in USERS:

    user_path = INPUT_PATH / user

    audio_files = sorted(user_path.glob("*.wav"))

    for audio_file in audio_files:

        signal, sample_rate = sf.read(audio_file)

        rms_features = librosa.feature.rms(
            y=signal,
            **RMS_CONFIG
        )

        rms_mean = np.mean(rms_features)
        rms_std = np.std(rms_features)

        rms_records.append({
            "User_ID": user,
            "Audio_Name": audio_file.name,
            "RMS_Mean": rms_mean,
            "RMS_Std": rms_std
        })

rms_df = pd.DataFrame(rms_records)

print("RMS Feature Extraction Completed")
print("-" * 45)

print("Total Recordings:", len(rms_df))
print("RMS Features per Recording:", 2)
print("Dataset Shape:", rms_df.shape)

display(rms_df.head())

# 6. Combine Voice Features

This section combines MFCC, ZCR, and RMS features into a single dataset.

The datasets are merged using User_ID and Audio_Name to ensure that features from the same audio recording remain correctly aligned.


User_ID will be used as the target label, while Audio_Name will be retained only as metadata.


In [ ]:
# Combine MFCC, ZCR, and RMS Features

voice_features_df = pd.merge(
    mfcc_df,
    zcr_df,
    on=["User_ID", "Audio_Name"],
    how="inner"
)

voice_features_df = pd.merge(
    voice_features_df,
    rms_df,
    on=["User_ID", "Audio_Name"],
    how="inner"
)

# Arrange Columns
metadata_columns = ["User_ID", "Audio_Name"]

feature_columns = [
    column
    for column in voice_features_df.columns
    if column not in metadata_columns
]

voice_features_df = voice_features_df[
    metadata_columns + feature_columns
]

print("Voice Feature Combination Completed")
print("-" * 45)

print("Total Recordings:", len(voice_features_df))
print("Total Numerical Features:", len(feature_columns))
print("Dataset Shape:", voice_features_df.shape)

print("\nFeature Distribution:")
print("MFCC Features:", len(mfcc_df.columns) - 2)
print("ZCR Features:", len(zcr_df.columns) - 2)
print("RMS Features:", len(rms_df.columns) - 2)

print("\nMissing Values:")
print(voice_features_df.isnull().sum().sum())

display(voice_features_df.head())

In [ ]:
# Create Output Directory

OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

# Save Combined Features

COMBINED_FEATURES_PATH = (
    OUTPUT_PATH / "Voice_Combined_Features.csv"
)

voice_features_df.to_csv(
    COMBINED_FEATURES_PATH,
    index=False
)

print("Voice Features Saved Successfully")
print("-" * 45)

print("File Path:", COMBINED_FEATURES_PATH)
print("Dataset Shape:", voice_features_df.shape)